In [0]:
%pip install python-dotenv -q
dbutils.library.restartPython()

In [0]:
%restart_python

In [0]:
# ==========================================================
#  PARAMETROS  - o unico bloco a alterar para trocar de tabela
# ==========================================================
SQUAD          = "squad2"
TABELA         = "ecommerce_pedidos"
CHAVE_PRIMARIA = "id_pedido"

# Origem: o gerador de tempo real grava um lote a cada ~120s em
# raw/real-time-data/AAAA/MM/DD/HHMMSS/<tabela>.parquet
CONTAINER      = "raw"
PREFIXO        = "real-time-data"
# Um nivel de glob por componente do carimbo (AAAA/MM/DD/HHMMSS):
# le todas as janelas de uma vez, sem listar diretorio.
PADRAO_LOTES   = "*/*/*/*"

# Schema da origem com as colunas de data como inteiro. So e usado se o
# Spark recusar o TIMESTAMP(NANOS) do parquet do gerador - ver 01, secao 1.
SCHEMA_DT_INTEIRO = """
    id_pedido bigint, id_cliente bigint, id_endereco_entrega bigint,
    dt_pedido bigint, status_pedido string, valor_total double, valor_frete double,
    metodo_pagamento string, dt_previsao_entrega bigint, dt_ultima_atualizacao_status bigint
"""
PREFIXO_DATA   = "dt_"

# Destino no Azure SQL Server
TABELA_DESTINO = f"{SQUAD}.{TABELA}"
MODO_ESCRITA   = "overwrite"
# ==========================================================

print(f"Origem : {CONTAINER}/{PREFIXO}/{PADRAO_LOTES}/{TABELA}.parquet")
print(f"Chave  : {CHAVE_PRIMARIA}")
print(f"Destino: {TABELA_DESTINO}  (modo {MODO_ESCRITA})")

In [0]:
import os
from pathlib import Path
from dotenv import dotenv_values

REPO = "estagio-empregadados-turma-2"


def descobrir_env():
    """Localiza o .env sem depender do e-mail de quem clonou o repositorio."""
    candidatos = []

    # 1. Subindo a partir do diretorio do proprio notebook.
    try:
        aqui = Path(os.getcwd()).resolve()
        candidatos += [p / ".env" for p in [aqui, *aqui.parents]]
    except Exception:
        pass

    # 2. /Workspace/Repos/<qualquer-usuario>/<repo>/.env
    raiz = Path("/Workspace/Repos")
    if raiz.is_dir():
        candidatos += sorted(raiz.glob(f"*/{REPO}/.env"))

    # 3. Locais avulsos.
    candidatos += [Path("/Workspace/Shared") / REPO / ".env"]

    for c in candidatos:
        try:
            if c.is_file():
                return c
        except Exception:
            continue
    return None


caminho_env = descobrir_env()
if caminho_env is None:
    raise FileNotFoundError(
        "Nenhum .env encontrado.\n"
        "Copie o .env.example da raiz do repositorio para .env e preencha os valores."
    )

CONFIG = {k: v for k, v in dotenv_values(caminho_env).items() if v}
print(f"[ok] .env carregado de: {caminho_env}")


def exigir(chaves, contexto):
    """Falha cedo e nomeia exatamente o que falta."""
    faltando = [c for c in chaves if not CONFIG.get(c)]
    if faltando:
        raise ValueError(f"{contexto}: variaveis ausentes no .env -> {', '.join(faltando)}")


exigir(["ADLS_STORAGE_ACCOUNT", "ADLS_CLIENT_ID", "ADLS_TENANT_ID", "ADLS_CLIENT_SECRET"], "ADLS Gen2")
print("[ok] credenciais do ADLS presentes")

OBRIGATORIAS_SQL = ["JDBC_HOSTNAME", "JDBC_DATABASE", "JDBC_USERNAME", "JDBC_PASSWORD"]
try:
    exigir(OBRIGATORIAS_SQL, "Azure SQL Server")
    print("[ok] credenciais do SQL Server presentes")
except ValueError as e:
    # 01 e 02 rodam sem SQL Server; so o 03 precisa.
    print(f"[aviso] {e}")

In [0]:
STORAGE_ACCOUNT = CONFIG["ADLS_STORAGE_ACCOUNT"]
_sufixo = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"

adls_options = {
    f"fs.azure.account.auth.type.{_sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{_sufixo}":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{_sufixo}": CONFIG["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{_sufixo}": CONFIG["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{_sufixo}":
        f"https://login.microsoftonline.com/{CONFIG['ADLS_TENANT_ID']}/oauth2/token",
}

CAMINHO_ORIGEM = f"abfss://{CONTAINER}@{_sufixo}/{PREFIXO}/{PADRAO_LOTES}/{TABELA}.parquet"


def opcoes_sqlserver(dbtable):
    """Opcoes do conector nativo `sqlserver` do Databricks."""
    exigir(OBRIGATORIAS_SQL, "Azure SQL Server")
    return {
        "host"    : CONFIG["JDBC_HOSTNAME"],
        "port"    : CONFIG.get("JDBC_PORT", "1433"),
        "database": CONFIG["JDBC_DATABASE"],
        "user"    : CONFIG["JDBC_USERNAME"],
        "password": CONFIG["JDBC_PASSWORD"],
        "dbtable" : dbtable,
    }


print("[ok] setup concluido - adls_options, CAMINHO_ORIGEM e opcoes_sqlserver() disponiveis")